In [1]:
import torch
from damage_upload import SegFormerDamageModel
import os
import torch
import rasterio
import numpy as np
from rasterio.merge import merge
from rasterio.windows import Window


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = SegFormerDamageModel()
model = model.to(device)

model.load_state_dict(torch.load("best_model.pth"))
model.eval()

Device: cuda


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b2-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([2, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SegFormerDamageModel(
  (model): SegformerForSemanticSegmentation(
    (segformer): SegformerModel(
      (encoder): SegformerEncoder(
        (patch_embeddings): ModuleList(
          (0): SegformerOverlapPatchEmbeddings(
            (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
            (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          )
          (1): SegformerOverlapPatchEmbeddings(
            (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
            (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          )
          (2): SegformerOverlapPatchEmbeddings(
            (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
            (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          )
          (3): SegformerOverlapPatchEmbeddings(
            (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          

In [4]:
def add_nodata_fill(input_file, output_file, tile_size):
    with rasterio.open(input_file) as src:
        data = src.read()
        profile = src.profile
        transform = src.transform
        width, height = src.width, src.height     

        fill_x = (width // tile_size + 1) * tile_size - width
        fill_y = (height // tile_size + 1)* tile_size - height

        new_data = np.zeros((src.count, height + fill_y, width + fill_x))
        new_data[:, :height, :width] = data
        with rasterio.open(
            output_file, 'w', driver='GTiff',
            width=width + fill_x, height=height + fill_y, count=src.count,
            dtype=src.dtypes[0], crs=src.crs, transform=transform, compress='LZW'
            ) as dst:
            dst.write(new_data)
            
def crop_landsat_image_to_geotiff(input_path, input_image_name, output_folder, tile_size):
    os.makedirs(output_folder,exist_ok=True)
    input_image = input_path + '/' + input_image_name
    with rasterio.open(input_image) as src:
        width, height = src.width, src.height

        for i in range(0, width, tile_size):
            for j in range(0, height, tile_size):
                window = Window(i, j, tile_size, tile_size)
                window_data = src.read(window=window)
                    
                window_transform = src.window_transform(window)  
                output_tile_path = "%s/%s_%s_%d_%d.tif" % (output_folder, input_image_name[:-4], "tiles", i/tile_size, j/tile_size)
                    
                with rasterio.open(
                    output_tile_path, 'w', driver='GTiff',
                    width=tile_size, height=tile_size, count=src.count,
                    dtype=src.dtypes[0], crs=src.crs, transform=window_transform
                ) as dst:
                    dst.write(window_data)

# Process the image and input it into the model for prediction
def process_image(image_path, model, device):
    with rasterio.open(image_path) as src:
        image = src.read().astype(np.uint8)
        image = torch.from_numpy(image).float() / 255.0
        image = image.unsqueeze(0)  
        profile = src.profile.copy()    
        image = image.to(device)

    with torch.no_grad():
        prediction = model(image)
        prediction = prediction.argmax(dim=1).to(torch.uint8).cpu()    # [1,H,W] uint8
        profile.update(count=1, dtype='uint8')   
    return prediction, profile

def save_geotiff(prediction, profile, output_path):
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write((prediction.cpu().numpy()* 255).astype(np.uint8))

def segment_geotiffs(input_folder, output_folder, model, device):
    
    os.makedirs(output_folder, exist_ok=True)
    for filename in os.listdir(input_folder):
        if filename.endswith('.tif') or filename.endswith('.tiff'):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            print(f"Processing {input_path}...")
            
            prediction, profile = process_image(input_path, model, device)
            
            save_geotiff(prediction, profile, output_path)
            print(f"Saved result to {output_path}")

def mosaic_geotiffs(input_folder, output_path):
    geotiff_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) 
                     if f.endswith('.tif') or f.endswith('.tiff')]
    
    src_files_to_mosaic = []
    for file in geotiff_files:
        src = rasterio.open(file)
        src_files_to_mosaic.append(src)
    
    mosaic, out_transform = merge(src_files_to_mosaic)
    
    out_meta = src_files_to_mosaic[0].meta.copy()
    
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform
    })
    
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(mosaic)
    
    for src in src_files_to_mosaic:
        src.close()
    
    print(f"Mosaic saved to {output_path}")

In [5]:
from __future__ import annotations
from typing import Tuple, Optional
import numpy as np
import torch
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window

# --------------------
# Configs
# --------------------
ROI_SIZE: Tuple[int, int] = (512, 512)   # patch size
OVERLAP: float = 0.5                     # 50% overlap
PADDING_MODE: str = "constant"           # Constant padding with 0 (can be changed to "reflect")
USE_TTA: bool = True                     # Mirror TTA (horizontal / vertical / both axes)
DEVICE: str = "cuda"                     # Device
READ_DTYPE = np.uint8                    # Read as uint8 
FOREGROUND_CHANNEL: int = 1              # Foreground channel index (used for saving foreground probabilities)

# --------------------
def gaussian_importance_map(h: int, w: int) -> np.ndarray:
    y = np.arange(h, dtype=np.float32)
    x = np.arange(w, dtype=np.float32)
    cy, cx = (h - 1) / 2.0, (w - 1) / 2.0
    sy, sx = max(h / 8.0, 1e-6), max(w / 8.0, 1e-6)
    yy = ((y - cy) / sy)[:, None] ** 2
    xx = ((x - cx) / sx)[None, :] ** 2
    m = np.exp(-0.5 * (yy + xx)).astype(np.float32)
    m /= (m.max() + 1e-6)

    w_floor = 1e-3  # 可调：1e-3 ~ 1e-2
    m = np.clip(m, w_floor, 1.0)
    return m


def pad_to_roi(arr: np.ndarray, target_h: int, target_w: int, mode: str = "constant"):
    _, H, W = arr.shape
    dh, dw = max(0, target_h - H), max(0, target_w - W)
    if dh == 0 and dw == 0:
        return arr, (0, 0, 0, 0)
    t = dh // 2; b = dh - t
    l = dw // 2; r = dw - l
    np_mode = {"reflect":"reflect","replicate":"edge","edge":"edge"}.get(mode, "constant")
    arr_pad = np.pad(arr, ((0,0),(t,b),(l,r)), mode=np_mode)  # constant=0
    return arr_pad, (t, b, l, r)

@torch.no_grad()
def _mirror_tta(predictor):
    """Return: [N, K, H, W] after averaging the probabilities from the four mirrored views"""
    def _pred(x: torch.Tensor) -> torch.Tensor:
        outs = []
        outs.append(predictor(x))
        outs.append(torch.flip(predictor(torch.flip(x, dims=[-1])), dims=[-1]))         # LR
        outs.append(torch.flip(predictor(torch.flip(x, dims=[-2])), dims=[-2]))         # UD
        outs.append(torch.flip(predictor(torch.flip(x, dims=[-2,-1])), dims=[-2,-1]))   # LR+UD
        return torch.stack(outs, 0).mean(0)  # [N,K,H,W]
    return _pred

@torch.no_grad()
def _mirror_rotate_tta(predictor):
    def _pred(x: torch.Tensor) -> torch.Tensor:
        outs = []
        flip_sets = [None, [-1], [-2], [-2, -1]]  # None, LR, UD, LR+UD
        for k in range(4):  # 0°, 90°, 180°, 270°
            x_rot = torch.rot90(x, k, dims=[-2, -1])
            for dims in flip_sets:
                x_aug = torch.flip(x_rot, dims=dims) if dims is not None else x_rot
                y = predictor(x_aug)
                if dims is not None:
                    y = torch.flip(y, dims=dims)
                y = torch.rot90(y, -k, dims=[-2, -1])
                outs.append(y)
        return torch.stack(outs, 0).mean(0)  # [N,K,H,W]
    return _pred

@torch.no_grad()
def sliding_window_infer_geotiff(
    model: torch.nn.Module,
    in_tif: str,
    out_label_tif: str,
    out_fgprob_tif: Optional[str] = None,   
    out_bgprob_tif: Optional[str] = None,
    roi_size: Tuple[int,int] = ROI_SIZE,
    overlap: float = OVERLAP,
    padding_mode: str = PADDING_MODE,
    use_tta: bool = USE_TTA,
    read_dtype = READ_DTYPE,
    device: str = DEVICE,
    foreground_channel: int = FOREGROUND_CHANNEL,
):
    assert 0 <= overlap < 1.0, "overlap must be in [0,1)"
    tile_h, tile_w = int(roi_size[0]), int(roi_size[1])
    assert tile_h > 0 and tile_w > 0

    model.eval().to(device)

    w2d = gaussian_importance_map(tile_h, tile_w)                 # [H,W]
    w2d_t = torch.from_numpy(w2d)[None, None].to(device)          # [1,1,H,W]

    # Predictor: binary 2-channel logits -> 2-channel probabilities [bg, fg]
    def predictor(x: torch.Tensor) -> torch.Tensor:
        logits = model(x)                                         # [N,2,H,W]
        C = logits.shape[1]
        if C != 2:
            raise RuntimeError(f"Model outputs {C} channels; expected 2 for binary [bg,fg].")
        return F.softmax(logits, dim=1)                           # [N,2,H,W]

    if use_tta:
        predictor = _mirror_tta(predictor)   #_mirror_tta(predictor)   _mirror_rotate_tta(predictor) 

    stride_h = max(1, int(round(tile_h * (1.0 - overlap))))
    stride_w = max(1, int(round(tile_w * (1.0 - overlap))))

    def _starts(L: int, win: int, st: int):

        if L <= win:
            return [0] 
        pts = list(range(0, L - win + 1, st))
        if pts[-1] != L - win:
            pts.append(L - win)  
        return pts


    with rasterio.open(in_tif, "r") as src:
        H, W = src.height, src.width
        read_bands = list(range(1, src.count + 1))              

        y0, x0 = 0, 0
        y1, x1 = min(H, tile_h), min(W, tile_w)
        probe = src.read(indexes=read_bands,
                         window=Window.from_slices((y0, y1), (x0, x1)),
                         out_dtype=read_dtype)                     # [C,h,w], uint8
        probe, _ = pad_to_roi(probe, tile_h, tile_w, padding_mode)
        probe = probe.astype(np.float32) / 255.0
        probe_t = torch.from_numpy(probe).unsqueeze(0).to(device)  # [1,C,H,W]
        out_probe = model(probe_t)                                  # [1,2,H,W]
        if out_probe.shape[1] != 2:
            raise RuntimeError(f"Warmup: model outputs {out_probe.shape[1]} channels; expected 2.")
        K = 2

        prob_acc = np.zeros((K, H, W), dtype=np.float32)
        wsum_acc = np.zeros((H, W), dtype=np.float32)

        ys = _starts(H, tile_h, stride_h)
        xs = _starts(W, tile_w, stride_w)

        for y in ys:
            for x in xs:
                y1, x1 = min(H, y + tile_h), min(W, x + tile_w)
                tile = src.read(indexes=read_bands,
                                window=Window.from_slices((y, y1), (x, x1)),
                                out_dtype=read_dtype)             # [C,h,w], uint8
                tile, (pt, pb, pl, pr) = pad_to_roi(tile, tile_h, tile_w, padding_mode)

                tile = tile.astype(np.float32) / 255.0            # [C,H,W]

                xt = torch.from_numpy(tile).unsqueeze(0).to(device)  # [1,C,H,W]
                probs = predictor(xt)                                # [1,2,H,W] 
                
                probs = probs.squeeze(0).detach().cpu().numpy()  # [2,H,W]

                ylen, xlen = (y1 - y), (x1 - x)
                probs_crop = probs[:, pt:pt+ylen, pl:pl+xlen]       # [2,ylen,xlen]
                w_crop = w2d[pt:pt+ylen, pl:pl+xlen]                # [ylen,xlen]

                prob_acc[:, y:y+ylen, x:x+xlen] += probs_crop * w_crop[None, ...]
                wsum_acc[y:y+ylen, x:x+xlen] += w_crop

        eps = 1e-6
        prob_fused = prob_acc / np.clip(wsum_acc[None, ...], eps, None)   # [2,H,W] in [0,1]
        label_map = np.argmax(prob_fused, axis=0).astype(np.uint8)        # [H,W] 0/1  Take the class with the maximum probability as the result



        mask_uncovered = (wsum_acc <= 0)
        ratio = float(mask_uncovered.sum()) / (wsum_acc.size)
        print("uncovered ratio:", ratio)


        # Save: label map 0/255
        profile = src.profile.copy()
        profile.update({"count": 1, "dtype": "uint8", "compress": "LZW"})
        with rasterio.open(out_label_tif, "w", **profile) as dst:
            dst.write((label_map * 255).astype(np.uint8), 1)

        # Save: foreground probability (optional)
        if out_fgprob_tif is not None:
            fg_prob = np.clip(prob_fused[1], 0.0, 1.0)
            # fg_255 = (fg_prob * 255.0 + 0.5).astype(np.uint8)             
            prof_p = src.profile.copy()
            prof_p.update({"count": 1, "dtype": "float32", "compress": "LZW"})
            with rasterio.open(out_fgprob_tif, "w", **prof_p) as dst:
                dst.write(fg_prob, 1)

        # Save: background probability (optional)
        if out_bgprob_tif is not None:
            bg_prob = np.clip(prob_fused[0], 0.0, 1.0)
            prof_b = src.profile.copy()
            prof_b.update({"count": 1, "dtype": "float32", "compress": "LZW"})
            with rasterio.open(out_bgprob_tif, "w", **prof_b) as dst:
                dst.write(bg_prob, 1)

        print(in_tif)
        
        print("IN-MEM sum:", float((prob_fused[0]+prob_fused[1]).min()),
                      float((prob_fused[0]+prob_fused[1]).max()),
                      float(np.mean(np.abs(prob_fused[0]+prob_fused[1]-1))))



In [ ]:
folder = r"E:\damage_code"    # Input image folder path
input_image_name= "LC08_Crosson_composite_2018-02-06_uint8.tif" # Input image name

input_image_path = os.path.join(folder,input_image_name)
mosaic_pred_path =  os.path.join(folder, input_image_name[:-4]+"_pred_sliding.tif")
mosaic_fgprob_path =  os.path.join(folder, input_image_name[:-4]+"_pred_sliding_fgprob.tif")
mosaic_bgprob_path =  os.path.join(folder, input_image_name[:-4]+"_pred_sliding_bgprob.tif")

sliding_window_infer_geotiff(
    model=model,
    in_tif=input_image_path,
    out_label_tif=mosaic_pred_path,
    out_fgprob_tif=None,  
    out_bgprob_tif=None,  
    roi_size=ROI_SIZE,
    overlap=OVERLAP,                   
    padding_mode=PADDING_MODE,
    use_tta=USE_TTA,                  
    read_dtype=READ_DTYPE,
    device=DEVICE,
    foreground_channel=FOREGROUND_CHANNEL,
)

uncovered ratio: 0.0
E:\damage_code\LC08_Crosson_composite_2018-02-06_uint8.tif
IN-MEM sum: 0.9999995231628418 1.0000004768371582 4.472978076819345e-08
